In [ ]:
import random
import datetime

def get_random_weekday(year):
    """
    Returns a random date from the given year that is not a Saturday or Sunday.
    """
    # Find the first and last day of the year
    start_date = datetime.date(year, 1, 1)
    end_date = datetime.date(year, 12, 31)
    # Generate all weekdays in the year
    weekdays = [
        start_date + datetime.timedelta(days=i)
        for i in range((end_date - start_date).days + 1)
        if (start_date + datetime.timedelta(days=i)).weekday() < 5  # 0=Mon, ..., 4=Fri
    ]
    # Pick one randomly
    return random.choice(weekdays)

In [ ]:
get_random_weekday(2023)

datetime.date(2023, 2, 28)

: 

#### Alternative way to get MLFLOW artifacts by using http through url

In [5]:
import requests
import os

run_id = "d069ee844a6540bdb2d04d14aa7c1db1"
# The exact endpoint MLflow uses to proxy files
url = f"http://localhost:5001/get-artifact?run_id={run_id}&path=run_model/scalers.pkl"

save_path = "./scalers.pkl"

print(f"Downloading directly from: {url}")
response = requests.get(url, stream=True)

if response.status_code == 200:
    with open(save_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("✅ Success! Scaler downloaded via direct HTTP proxy.")
else:
    print(f"❌ Failed. Status: {response.status_code}")
    print(f"Response: {response.text}")

✅ Success! Scaler downloaded via direct HTTP proxy.


#### Using GetData to get the data for a day

In [ ]:
import rl_functions.utils as rl_utils

from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd().parent / '.env'
print(f"env_path: {env_path}")
load_dotenv(dotenv_path=env_path, override=True)
print(os.getenv('POSTGRES_HOST'))
from trading_functions.db.session import SessionLocal
db = SessionLocal()
def get_env(primary: str, fallback: str, default: str) -> str:
    """Get env var, preferring `primary` over `fallback`."""
    return os.getenv(primary) or os.getenv(fallback, default)
POSTGRES_HOST = get_env("POSTGRES_HOST", "DAGSTER_POSTGRES_HOST", "localhost")
print(POSTGRES_HOST)
### Getting the date 
import datetime
#date_str = "2023-01-04"
#datetime_var = datetime.strptime(date_str, "%Y-%m-%d")
START_DATE = datetime.date(2023, 1, 5)
END_DATE = datetime.date(2023, 1, 6)
data_df, status = rl_utils.get_data(symbol="SPY", start_date=START_DATE, end_date=END_DATE, db=db, evaluation=True)
data_df[['Date','pred_high','pred_low', 'pred_high_diff', 'pred_low_diff', 'pred_high_error', 'pred_low_error']].tail()
#data_df.tail()

2026-04-18 18:49:58,517 - rl_functions.utils - INFO - Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev_local.yaml
2026-04-18 18:49:58,538 - rl_functions.utils - INFO - Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev_local.yaml


2026-04-18 18:49:58,557 - root - INFO - Using default MLflow tracking URI from config : http://localhost:5001
2026-04-18 18:49:58,561 - root - INFO - Using model alias: rl for model: xgboost_rob_2023_jan_high
2026-04-18 18:49:58,583 - root - INFO - scaler path: run_model/scalers.pkl
2026-04-18 18:49:58,584 - root - INFO - config artifact path: run_model/training_config.yaml
2026-04-18 18:49:58,585 - root - INFO - Run ID: 3c97bd2da876481d978de23997d0c42e
2026-04-18 18:49:58,585 - root - INFO - Deleting existing scaler directory: /model_local_artifacts
2026-04-18 18:49:58,637 - root - INFO - Downloading artifacts to /model_local_artifacts


env_path: c:\Projects\Trading\Uns_SPY_Trading\.env
localhost
localhost
Evaluation mode: ON
run_model/scalers.pkl  (dir: False)
run_model/training_config.yaml  (dir: False)


2026-04-18 18:49:58,702 - root - INFO - Scalers downloaded to c:\model_local_artifacts\scalers.pkl
2026-04-18 18:49:58,706 - root - INFO - Scalers loaded: dict_keys(['minmax', 'standard', 'robust'])
2026-04-18 18:49:58,905 - root - INFO - Config downloaded to c:\model_local_artifacts\training_config.yaml
2026/04/18 18:49:58 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pandas (current: 2.3.3, required: pandas==2.3.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/04/18 18:49:59 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pandas (current: 2.3.3, required: pandas==2.3.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the m

Data shape after transformation: (489, 74)
                   Date      Open     High      Low    Close  Volume  \
99  2023-01-03 14:21:00  381.6759  381.760  381.545  381.635   46290   
100 2023-01-03 14:22:00  381.6400  381.750  381.640  381.690   39773   
101 2023-01-03 14:23:00  381.6801  381.765  381.390  381.425   74290   
102 2023-01-03 14:24:00  381.4100  381.440  381.190  381.290   85884   
103 2023-01-03 14:25:00  381.2900  381.370  381.140  381.150   42123   

         MA20      MA50     MA100     EMA20  ...  fourier_imag_24  Volatility  \
99   0.416261  0.372448 -0.036208  0.477549  ...         0.809430    0.137799   
100  0.284160  0.299019 -0.079187  0.316344  ...        -0.970557    0.146246   
101  0.804980  0.636437  0.147398  0.901223  ...         0.983255    0.308270   
102  1.024326  0.806308  0.262739  1.133809  ...        -1.093200    0.302514   
103  1.251218  0.975001  0.383674  1.355869  ...         1.001260    0.325428   

     Momentum      PVPT    PVPTR1    

,Date,pred_high,pred_low,pred_high_diff,pred_low_diff,pred_high_error,pred_low_error
385,2023-01-04 15:55:00,384.006207,380.773949,0.836207,2.396051,0.836207,-0.442351
386,2023-01-04 15:56:00,383.989156,380.230279,0.789156,2.969721,0.779156,-0.986021
387,2023-01-04 15:57:00,383.989248,380.055302,0.589248,3.344698,0.589248,-1.160998
388,2023-01-04 15:58:00,383.999465,380.588546,0.499465,2.911454,0.489465,-0.627754
389,2023-01-04 15:59:00,384.144286,381.104511,0.464286,2.575489,0.154286,-0.111789


#### Using RLStream to get the same data

In [3]:
from trading_functions.inference.RL_Streamer import RL_Stream_Data
from trading_functions.db.session import SessionLocal
import yaml
from datetime import datetime, timedelta
import logging, sys
from dotenv import load_dotenv

# Load env variables
load_dotenv("../.env_local", override=True)

#### CReate a logger
logger = logging.getLogger("Notebook_Logger")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
if not logger.hasHandlers():
    logger.addHandler(handler)

# TEst the logger
logger.info("Logger is set up and working.")
db = SessionLocal()
config_path = "../Config/config_dev_local.yaml"
with open(config_path, 'r') as file:
    conf = yaml.safe_load(file)


streamer = RL_Stream_Data(
    db=db,
    symbol='SPY',
    inf_config=conf,
    simulation_mode=True,
    model_high_version=1,
    model_low_version=1,
    model_high_alias='rl',
    start_time=datetime(2023,1,4),
    end_time=datetime(2023,1,5),
    logger=logger
)



2026-04-18 18:30:12,382 - Notebook_Logger - INFO - Logger is set up and working.
2026-04-18 18:30:12,400 - Notebook_Logger - INFO - Initialized RL_Stream_Data with symbol: SPY, simulation_mode: True, start_time: 2023-01-04 00:00:00, end_time: 2023-01-05 00:00:00
2026-04-18 18:30:12,401 - Notebook_Logger - INFO - Loading models and scalers for xgboost model high version: 1 , model low version: 1 , high alias: rl
2026-04-18 18:30:12,402 - rl_functions.utils - INFO - Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev_local.yaml
2026-04-18 18:30:12,416 - root - INFO - Using default MLflow tracking URI from config : http://localhost:5001
2026-04-18 18:30:12,419 - root - INFO - Using model alias: rl for model: xgboost_rob_2023_jan_high
2026-04-18 18:30:12,440 - root - INFO - scaler path: run_model/scalers.pkl
2026-04-18 18:30:12,442 - root - INFO - config artifact path: run_model/training_config.yaml
2026-04-18 18:30:12,442 - root - INFO - Run ID: 3c97

run_model/scalers.pkl  (dir: False)
run_model/training_config.yaml  (dir: False)


2026-04-18 18:30:12,787 - root - INFO - Scalers downloaded to c:\model_local_artifacts\scalers.pkl
2026-04-18 18:30:12,795 - root - INFO - Scalers loaded: dict_keys(['minmax', 'standard', 'robust'])
2026-04-18 18:30:12,919 - root - INFO - Config downloaded to c:\model_local_artifacts\training_config.yaml
2026/04/18 18:30:12 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pandas (current: 2.3.3, required: pandas==2.3.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/04/18 18:30:13 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pandas (current: 2.3.3, required: pandas==2.3.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the m

In [4]:

stream_data = streamer.stream()
stream_rows = []
for n in range(10):
    stream_rows.append(next(stream_data))
stream_rows

2026-04-18 18:35:21,126 - Notebook_Logger - INFO - Previous date entry found: 2023-01-03


Warming up current day history buffer...


2026-04-18 18:35:22,063 - Notebook_Logger - INFO - Initial day_total_bars: 390


Data shape after transformation: (99, 74)
                   Date      Open     High      Low    Close  Volume  \
99  2023-01-03 14:21:00  381.6759  381.760  381.545  381.635   46290   
100 2023-01-03 14:22:00  381.6400  381.750  381.640  381.690   39773   
101 2023-01-03 14:23:00  381.6801  381.765  381.390  381.425   74290   
102 2023-01-03 14:24:00  381.4100  381.440  381.190  381.290   85884   
103 2023-01-03 14:25:00  381.2900  381.370  381.140  381.150   42123   

         MA20      MA50     MA100     EMA20  ...  fourier_imag_24  Volatility  \
99   0.416261  0.372448 -0.036208  0.477549  ...         0.809430    0.137799   
100  0.284160  0.299019 -0.079187  0.316344  ...        -0.970557    0.146246   
101  0.804980  0.636437  0.147398  0.901223  ...         0.983255    0.308270   
102  1.024326  0.806308  0.262739  1.133809  ...        -1.093200    0.302514   
103  1.251218  0.975001  0.383674  1.355869  ...         1.001260    0.325428   

     Momentum      PVPT    PVPTR1    P

[Date               2023-01-04 09:30:00
 Open                            383.18
 High                            383.24
 Low                             382.82
 Close                          382.871
                           ...         
 pred_high_error               0.369095
 pred_low_error               -0.414545
 momentum_short                0.000567
 velocity_short               -0.000123
 step_progress                 0.002564
 Name: 99, Length: 95, dtype: object,
 Date               2023-01-04 09:31:00
 Open                            382.88
 High                            383.28
 Low                             382.61
 Close                           382.67
                           ...         
 pred_high_error               0.086786
 pred_low_error               -0.784779
 momentum_short                0.000366
 velocity_short               -0.000201
 step_progress                 0.005128
 Name: 100, Length: 95, dtype: object,
 Date               2023-01-04 09:32:00
 Op